In [1]:
# =====================================================
# Master Flight Operations Dataset Integration
# =====================================================

import pandas as pd
import numpy as np

from sklearn.neighbors import BallTree

import warnings
warnings.filterwarnings("ignore")

pd.set_option("display.max_columns", None)

C:\Users\Darshan V\AppData\Roaming\Python\Python313\site-packages\pandas\core\computation\expressions.py:23: UserWarning: Pandas requires version '2.10.2' or newer of 'numexpr' (version '2.10.1' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED


In [4]:
# =====================================================
# Load Processed Datasets
# =====================================================

opensky = pd.read_csv("../data/processed/opensky/opensky_feature_engineered.csv")

airports = pd.read_csv("../data/processed/airports/airports_feature_engineered.csv")

weather = pd.read_csv("../data/processed/weather/weather_feature_engineered.csv")

aircraft = pd.read_csv("../data/processed/aircraft/aircraft_metadata_feature_engineered.csv")

delay = pd.read_csv("../data/processed/delays/delays_feature_engineered.csv")

In [5]:
datasets = {
    "OpenSky": opensky,
    "Airports": airports,
    "Weather": weather,
    "Aircraft": aircraft,
    "Flight Delay": delay
}

for name, df in datasets.items():

    print("="*60)
    print(name)
    print("="*60)

    print("Shape :", df.shape)
    print("\nColumns\n")
    print(df.columns.tolist())

    print("\nMissing Values\n")
    print(df.isnull().sum().sort_values(ascending=False).head(10))

OpenSky
Shape : (7516, 22)

Columns

['icao24', 'callsign', 'origin_country', 'time_position', 'last_constant', 'longitude', 'latitude', 'baro_altitude', 'on_ground', 'velocity', 'true_track', 'vertical_rate', 'geo_altitude', 'squawk', 'spi', 'position_source', 'flight_phase', 'speed_category', 'altitude_band', 'vertical_movement', 'heading_direction', 'operational_risk']

Missing Values

squawk            4286
callsign            16
origin_country       0
time_position        0
last_constant        0
icao24               0
longitude            0
latitude             0
on_ground            0
baro_altitude        0
dtype: int64
Airports
Shape : (85670, 22)

Columns

['id', 'ident', 'type', 'name', 'latitude_deg', 'longitude_deg', 'elevation_ft', 'continent', 'iso_country', 'iso_region', 'municipality', 'scheduled_service', 'icao_code', 'iata_code', 'gps_code', 'airport_size', 'airport_category', 'commercial_airport', 'elevation_category', 'hemisphere', 'climate_zone', 'airport_importanc

In [6]:
print("OpenSky ICAO24")

print(opensky["icao24"].nunique())

print()

print("Aircraft Mode-S")

print(aircraft["MODE S CODE HEX"].nunique())

OpenSky ICAO24
7514

Aircraft Mode-S
314417


In [7]:
opensky["icao24"] = (
    opensky["icao24"]
    .astype(str)
    .str.upper()
    .str.strip()
)

aircraft["MODE S CODE HEX"] = (
    aircraft["MODE S CODE HEX"]
    .astype(str)
    .str.upper()
    .str.strip()
)

In [8]:
master = opensky.merge(
    aircraft,
    how="left",
    left_on="icao24",
    right_on="MODE S CODE HEX"
)

In [9]:
print("Rows")

print(master.shape)

print()

print("Matched Aircraft")

print(master["MODE S CODE HEX"].notna().sum())

print()

print("Missing Aircraft")

print(master["MODE S CODE HEX"].isna().sum())

Rows
(7516, 59)

Matched Aircraft
4467

Missing Aircraft
3049


In [10]:
master.head()

,icao24,callsign,origin_country,time_position,last_constant,longitude,latitude,baro_altitude,on_ground,velocity,true_track,vertical_rate,geo_altitude,squawk,spi,position_source,flight_phase,speed_category,altitude_band,vertical_movement,heading_direction,operational_risk,N-NUMBER,SERIAL NUMBER,MFR MDL CODE,ENG MFR MDL,YEAR MFR,TYPE REGISTRANT,LAST ACTION DATE,CERT ISSUE DATE,CERTIFICATION,TYPE AIRCRAFT,TYPE ENGINE,STATUS CODE,MODE S CODE,AIR WORTH DATE,EXPIRATION DATE,UNIQUE ID,MODE S CODE HEX,MANUFACTURER,AIRCRAFT_MODEL,AIRCRAFT_TYPE_CODE,ENGINE_TYPE_CODE,AIRCRAFT_CATEGORY,BUILD_CERTIFICATE,NUMBER_OF_ENGINES,SEAT_CAPACITY,WEIGHT_CLASS,CRUISE_SPEED,TYPE_CERTIFICATE,CERTIFICATE_HOLDER,Aircraft_Age,Aircraft_Age_Category,Engine_Configuration,Seat_Capacity_Category,Weight_Category,Cruise_Speed_Category,Manufacturer_Group,Aircraft_Size_Category
0,80162D,AXB848,India,1.767745e+09,1767745048,52.7391,25.4425,9608.82,False,257.34,119.59,5.20,9829.80,3112.0,False,0,Climb,Cruise Speed,High Altitude,Climbing,East,Medium,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,AE1FA0,72209,United States,1.767745e+09,1767745047,-84.9380,38.1463,571.50,False,73.38,12.96,0.33,487.68,NaN,False,0,Cruise,Low Speed,Low Altitude,Level Flight,North,Low,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,AC96B8,AAL1175,United States,1.767745e+09,1767745048,-102.0238,34.0962,10363.20,False,196.90,291.13,0.00,10683.24,NaN,False,0,Cruise,Cruise Speed,Cruise Altitude,Level Flight,West,Low,910AN,29512,13844CB,13802.0,1999.0,3.0,2023-10-20,1999-05-27,1T,5,5.0,V,53113270.0,1999-05-25,2030-04-30,927071.0,AC96B8,BOEING,737-823,5,5.0,1.0,0.0,2.0,162.0,CLASS 3,0.0,,...,27.0,Mid-Life,Twin Engine,Medium Commercial,Heavy,Unknown,Commercial,Narrow-Body
3,C81BD3,ZKAMK,New Zealand,1.767745e+09,1767744917,174.5408,-35.8060,624.84,False,56.90,165.34,0.65,670.56,NaN,False,0,Cruise,Low Speed,Low Altitude,Level Flight,South,Low,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,AA56DB,UAL2447,United States,1.767745e+09,1767745048,-102.7402,37.4696,10972.80,False,200.08,327.49,0.00,11155.68,NaN,False,0,Cruise,Cruise Speed,Cruise Altitude,Level Flight,North,Low,76529,31652,13844BZ,13073.0,2010.0,3.0,2023-09-22,2010-12-29,1T,5,5.0,V,52453333.0,2010-12-23,2029-12-31,1060884.0,AA56DB,BOEING,737-824,5,5.0,1.0,0.0,2.0,149.0,CLASS 3,0.0,,...,16.0,Mid-Life,Twin Engine,Medium Commercial,Heavy,Unknown,Commercial,Narrow-Body


In [11]:
# ============================================
# Keep only airports with coordinates
# ============================================

airport_locations = airports[
    ["ident",
     "name",
     "municipality",
     "iso_country",
     "airport_size",
     "airport_category",
     "airport_importance",
     "latitude_deg",
     "longitude_deg"]
].copy()

airport_locations = airport_locations.dropna(
    subset=["latitude_deg", "longitude_deg"]
)

print(airport_locations.shape)
airport_locations.head()

(85670, 9)


,ident,name,municipality,iso_country,airport_size,airport_category,airport_importance,latitude_deg,longitude_deg
0,00A,Total RF Heliport,Bensalem,US,Heliport,Heliport,Low,40.070985,-74.933689
1,00AA,Aero B Ranch Airport,Leoti,US,Small,Local,Low,38.704022,-101.473911
2,00AK,Lowell Field,Anchor Point,US,Small,Local,Low,59.947733,-151.692524
3,00AL,Epps Airpark,Harvest,US,Small,Local,Low,34.864799,-86.770302
4,00AN,Katmai Lodge Airport,King Salmon,US,Small,Local,Low,59.093287,-156.456699


In [12]:
# ============================================
# Convert coordinates to radians
# ============================================

airport_locations["lat_rad"] = np.radians(
    airport_locations["latitude_deg"]
)

airport_locations["lon_rad"] = np.radians(
    airport_locations["longitude_deg"]
)

master["lat_rad"] = np.radians(master["latitude"])

master["lon_rad"] = np.radians(master["longitude"])

In [13]:
# ============================================
# Build BallTree
# ============================================

airport_tree = BallTree(
    airport_locations[["lat_rad", "lon_rad"]],
    metric="haversine"
)

In [14]:
# ============================================
# Find nearest airport
# ============================================

distances, indices = airport_tree.query(
    master[["lat_rad", "lon_rad"]],
    k=1
)

In [15]:
EARTH_RADIUS = 6371

master["distance_to_airport_km"] = (
    distances.flatten() * EARTH_RADIUS
)

In [16]:
nearest_airport = airport_locations.iloc[
    indices.flatten()
].reset_index(drop=True)

nearest_airport = nearest_airport.add_prefix("nearest_")

master = pd.concat(
    [master.reset_index(drop=True),
     nearest_airport],
    axis=1
)

In [17]:
print(master.shape)

master[
    [
        "icao24",
        "nearest_ident",
        "nearest_name",
        "nearest_municipality",
        "nearest_airport_importance",
        "distance_to_airport_km"
    ]
].head(10)

(7516, 73)


,icao24,nearest_ident,nearest_name,nearest_municipality,nearest_airport_importance,distance_to_airport_km
0,80162D,OMAS,Das Island Airport,Das Island,Low,36.466053
1,AE1FA0,05KT,High Point Farm Airport,Frankfort,Low,4.587900
2,AC96B8,1TX5,Laney Farm Airport,Hale Center,Low,11.098662
3,C81BD3,NZ-0085,Tahere Airstrip,Whareora,Low,15.415712
4,AA56DB,K8V7,Springfield Municipal Airport,Springfield,Low,10.853538
5,A2E5EC,US-4296,Rambla Pacifico Helipad,Malibu,Low,6.820989
6,A7B08D,17MU,B-B Airfield,Osborn,Low,8.286365
7,7C6B2D,NZFP,Foxpine Aerodrome,Unknown,Low,39.665468
8,88044F,TH-0034,TFT Airfield,Nong Suea,Low,7.663597
9,408121,CDU3,Yarmouth Regional Hospital Heliport,Yarmouth,Low,40.541261


In [18]:
airport_locations = airports[
    airports["commercial_airport"] == True
].copy()

In [19]:
print(airports["commercial_airport"].value_counts(dropna=False))

commercial_airport
0    81267
1     4403
Name: count, dtype: int64


In [20]:
print(airports["airport_size"].value_counts())

airport_size
Small       42656
Heliport    23097
Closed      13310
Medium       4097
Seaplane     1271
Large        1178
Balloon        61
Name: count, dtype: int64
